**Navigation** : [Version C#](Z3-13b-UnsatCores-MUS-CSharp.ipynb) | [Version Rust](Z3-13b-UnsatCores-MUS-Rust.ipynb) | ← [13 - UNSAT cores](Z3-13-UnsatCores-Python.ipynb) | [README Z3-Python](README.md) | [14 - Bit-vectors >>](Z3-14-BitVectors-Overflow-Python.ipynb)

# Z3-Python-13b — UNSAT cores : le MUS (sous-ensemble irréductible)

**Objectifs** : extraire un core et vérifier l’irréductibilité d’un MUS.  
**Prérequis** : contraintes Z3 et notebook 13. **Durée** : 40 minutes.

*Compagnon « preuve minimale » du module [Z3-Python-13 — UNSAT cores](Z3-13-UnsatCores-Python.ipynb).*

Le notebook [13](Z3-13-UnsatCores-Python.ipynb) a établi le **noyau d'insatisfiabilité** (core) : `assert_and_track` étiquette chaque contrainte, `unsat_core()` isole celles qui conflictent. Ce compagnon 13b pousse là où le 13 s'arrête : d'abord un système où le conflit est **invisible à l'inspection** (aucune paire ne se contredit, l'insatisfiabilité émerge d'une combinaison), ensuite la limite de `unsat_core()` — Z3 ne garantit **pas** la minimalité — et l'algorithme **deletion-based** qui calcule le **MUS** (Minimal Unsatisfiable Subset) : le sous-ensemble *irréductible*, dont retirer n'importe quelle contrainte rend le système satisfiable.
Le notebook [13](Z3-13-UnsatCores-Python.ipynb) a établi le **noyau d'insatisfiabilité** (core) : `assert_and_track` étiquette chaque contrainte, `unsat_core()` isole celles qui conflitent. Ce compagnon 13b pousse là où le 13 s'arrête : d'abord un système où le conflit est **invisible à l'inspection** (aucune paire ne se contredit, l'insatisfiabilité émerge d'une combinaison), ensuite la limite de `unsat_core()` — Z3 ne garantit **pas** la minimalité — et l'algorithme **deletion-based** qui calcule le **MUS** (Minimal Unsatisfiable Subset) : le sous-ensemble *irréductible*, dont retirer n'importe quelle contrainte rend le système satisfiable.

## Source

Le livre-ancrage de la série est **Yurichev, Dennis — *SAT/SMT by Example*** (édition publique du 12 février 2023, 666 pages) : §3.13.2 « Unsat core » (p. 90) et §22.1 « Minimal Unsatisfiable Subformula » (p. 590, chapitre 22 « Everything else »).

Le §3.13.2 pose l'extraction d'un core côté Z3 : `assert_and_track` étiquette les contraintes, `unsat_core()` isole celles qui se contredisent — son exemple est un tableur à dépendance circulaire `C0` ↔ `C1`, dont le core vaut `[C0, C1]` ; le §3.13.3 « Stress test » (p. 92) monte la charge sur un tableur aléatoire. Le §22.1 attaque le **MUS** : à partir du problème de Knuth (TAOCP 7.1.11), l'auteur fait calculer un MUS par PicoMUS — « computed MUS of size 8 out of 16 (50 %) » (p. 592) — et rappelle qu'un MUS est le sous-ensemble dont le retrait de toute clause rend l'instance satisfiable.

Le carnet **transporte le MUS au niveau des contraintes Z3** : il montre que `unsat_core()` ne garantit pas la minimalité, puis calcule le sous-ensemble irréductible par un algorithme **deletion-based**, sur une chaîne de contraintes dont l'insatisfiabilité n'apparaît dans aucune paire — le mécanisme de core est, lui, repris du carnet [13](Z3-13-UnsatCores-Python.ipynb).

Copie archivée : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\Satisfiability and SMT Solvers\2023 - SAT_SMT_by_example.pdf` (16 456 233 octets). Voir [Z3-API/README.md — Ouvrage de référence](README.md#ouvrage-de-référence) pour l'index chapitré complet de la série.


In [1]:
from z3 import *
import sys
print(f"Python {sys.version.split()[0]} ; Z3 {get_version_string()}")

Python 3.13.16 ; Z3 5.4.1


## 1. Le vrai test : un core invisible a l'inspection

Les exemples du notebook [13 - UNSAT cores](Z3-13-UnsatCores-Python.ipynb) (sections 2 a 5) partagent une limite : dans chacun, le conflit est **visible a l'oeil nu** -- deux contraintes se contredisent textuellement (`h == 12` vs `h != 12`, `x == 1` vs `x == 2`). On n'a pas vraiment besoin de Z3 pour les isoler.

C'est la que `unsat_core()` devient precieux : sur un systeme ou **aucune paire** de contraintes ne se contredit, mais ou l'insatisfiabilite **emerge d'une combinaison**. Cas concret, un projet de planification : quatre taches $T_0, T_1, T_2, T_3$ s'enchainent (chacune doit finir avant que la suivante commence), chacune dure 2 unites, et tout doit etre termine avant la date 7. Aucune contrainte prise isolement n'est absurde -- et pourtant le systeme est insatisfiable. Lequel des 11 ingredients faut-il relacher ?

In [2]:
# Planification surcontrainte : 4 taches enchainees (T0->T1->T2->T3), duree 2 chacune,
# a finir avant la date 7. Aucune contrainte isolee n'est absurde -> conflit structurel.
s_start = [Int(f"start_{i}") for i in range(4)]
DUREE = 2
DATE_LIMITE = 7
s5 = Solver()
s5.set("unsat_core", True)
labels = []

def ajouter(constraint, nom):
    b = Bool(nom)
    s5.assert_and_track(constraint, b)
    labels.append(b)
    return b

# Chaque tache : demarre apres 0 et finit avant la date limite
for i in range(4):
    ajouter(s_start[i] >= 0, f"release_T{i}")
    ajouter(s_start[i] + DUREE <= DATE_LIMITE, f"deadline_T{i}")
# Enchainement : Ti finit avant que Ti+1 commence
for i in range(3):
    ajouter(s_start[i] + DUREE <= s_start[i + 1], f"precedence_T{i}_vers_T{i + 1}")

print("Check : %s" % s5.check())
core = s5.unsat_core()
print("Core Z3 = { %s }" % "  ".join(str(c) for c in core))
print("Taille du core : %d contraintes sur %d" % (len(core), len(labels)))
innocentes = [str(b) for b in labels if b not in core]
print("Contraintes innocentes ecartees (%d) : %s" % (len(innocentes), ", ".join(innocentes)))

Check : unsat
Core Z3 = { release_T0  deadline_T3  precedence_T0_vers_T1  precedence_T1_vers_T2  precedence_T2_vers_T3 }
Taille du core : 5 contraintes sur 11
Contraintes innocentes ecartees (6) : deadline_T0, release_T1, deadline_T1, release_T2, deadline_T2, release_T3


### Lecture : le core isole la chaine, pas une paire

Le core observé contient **5 contraintes sur 11** : le `release_T0` (point de depart), les **trois precedences** qui enchainent $T_0 \to T_1 \to T_2 \to T_3$, et le `deadline_T3` (echeance finale). Les **6 autres** -- les bornes et echeances intermediaires `deadline_T0`, `release_T1`/`T2`/`T3`, `deadline_T1`/`T2` -- sont ecartees : la chaine les implique deja, elles ne sont pas necessaires au conflit.

Pourquoi ce core est **invisible a l'inspection** : aucune des 5 contraintes coupables ne contredit une autre textuellement. Il faut **tracer la chaine** -- `start_T0 >= 0`, puis `+2` a chaque precedence, donc `start_T1 >= 2`, `start_T2 >= 4`, `start_T3 >= 6`, puis `+2` de duree = `8 > 7` (la deadline) -- pour voir le conflit. Et ce core de 5 est le **SEUL possible** : retirer `release_T0` et la chaine flotte librement, retirer une precedence et la chaine se coupe, retirer `deadline_T3` et chaque echeance restante est tenable -- le core rendu n'est pas un choix parmi d'autres, c'est l'unique sous-ensemble irreductible, et les 6 contraintes ecartees le sont a bon droit.

C'est precisement le travail fastidieux qu'evite `unsat_core()` : parmi 11 contraintes, il isole chirurgicalement les 5 qui forment la preuve d'insatisfiabilite.



### Exercice 1 — Irréductibilité membre par membre
Dans les huit contraintes suivantes, choisissez les étiquettes d'un MUS. Le garde teste UNSAT et chaque retrait ; une réponse vide ou redondante est refusée.

**Étapes** : former un sous-ensemble candidat, tester son insatisfiabilité, puis retirer chaque membre à tour de rôle pour vérifier son caractère essentiel.

In [3]:
def verifier_mus(constraints, labels):
    noms = [str(label) for _, label in constraints]
    if labels is None or not labels or len(set(labels)) != len(labels) or any(x not in noms for x in labels):
        return False
    selection = [(f, e) for f, e in constraints if str(e) in labels]
    def statut(paires):
        solveur = Solver()
        solveur.add(*(f for f, _ in paires))
        return solveur.check()
    return statut(selection) == unsat and all(statut(selection[:i] + selection[i+1:]) == sat for i in range(len(selection)))
w = Int("w_ex1")
contraintes_ex1 = [(w >= 0, Bool("nonneg")), (w >= 4, Bool("lower")),
    (w + 4 <= 6, Bool("upper")), (w == 5, Bool("equal")),
    (w <= 100, Bool("bound")), (w != 7, Bool("not7")),
    (w >= -10, Bool("floor")), (w <= 50, Bool("ceiling"))]
reponse_ex1 = None  # TODO : liste des étiquettes d'un MUS
print("Exercice 1 à compléter" if reponse_ex1 is None else f"MUS valide : {verifier_mus(contraintes_ex1, reponse_ex1)}")

Exercice 1 à compléter


## 2. Au-dela du core : le MUS (Minimal Unsatisfiable Subset)

Jusqu'ici, on a fait confiance a `unsat_core()` pour renvoyer un sous-ensemble
**minimal** de contraintes. Mais Z3 ne le **garantit pas** : le core est
*un sous-ensemble responsable du conflit*, pas necessairement le plus petit.
Sur l'arithmetique lineaire simple, Z3 minimise souvent bien ; sur des modeles
plus riches (graphe de precedences, contraintes transitives), il peut laisser
des contraintes **redundantes** dans le core.

Le **MUS** (Minimal Unsatisfiable Subset) est le sous-ensemble *irreductible* :
retirer n'importe laquelle de ses contraintes rend le systeme **satisfiable**.
L'algorithme **deletion-based** est simple et garantit cette irreductibilite :
on retire les contraintes une a une, et si le systeme reste UNSAT sans l'une
d'elles, c'est qu'elle etait superflue — on la supprime definitivement.

Le **core** répond : « ces contraintes suffisent à établir UNSAT ». Le **MUS** ajoute : « aucune de ces contraintes ne peut être retirée sans retrouver SAT ». Cette garantie est une minimalité par inclusion, pas une recherche du cardinal minimum.


In [4]:
from z3 import *

def compute_mus(constraints):
    """MUS deletion-based : sous-ensemble irreductible rendant le systeme UNSAT.

    constraints : liste de couples (formule, etiquette Bool).
    Renvoie un sous-ensemble irreductible tel qu'aucune contrainte ne puisse
    etre retiree sans rendre le systeme satisfiable."""
    mus = list(constraints)
    initial = Solver()
    initial.add(*(f for f, _ in mus))
    status = initial.check()
    if status == sat:
        raise ValueError("Le système doit être insatisfiable")
    if status == unknown:
        raise RuntimeError("Z3 a renvoyé unknown")
    i = 0
    while i < len(mus):
        candidat = mus[:i] + mus[i+1:]          # mus sans la contrainte i
        s = Solver(); s.set("unsat_core", True)
        for formule, etiquette in candidat:
            s.assert_and_track(formule, etiquette)
        status = s.check()
        if status == unsat:
            mus = candidat                       # reste UNSAT -> i etait superflue
        elif status == sat:
            i += 1
        else:
            raise RuntimeError("Z3 a renvoyé unknown")                                # i est essentielle -> on la garde
    return mus

# Ordonnancement de taches : 5 taches t0..t4 avec precedences (durees) + deadline.
# Les aretes transitives (ex. t0 -> t3) rendent certaines contraintes redondantes.
t = [Int('t%d' % k) for k in range(5)]
precedences = [
    (t[1] >= t[0] + 5,  Bool('e01')),   # t0 -> t1 (duree 5)
    (t[2] >= t[1] + 3,  Bool('e12')),   # t1 -> t2 (duree 3)
    (t[3] >= t[2] + 4,  Bool('e23')),   # t2 -> t3 (duree 4)
    (t[3] >= t[0] + 12, Bool('e03')),   # t0 -> t3 transitif (5+3+4 = 12)
    (t[4] >= t[3] + 2,  Bool('e34')),   # t3 -> t4 (duree 2)
    (t[4] <= t[0] + 10, Bool('dead')),  # deadline : chemin total = 14 > 10 -> UNSAT
]

# Core brut de Z3
s_core = Solver(); s_core.set("unsat_core", True)
for f, e in precedences:
    s_core.assert_and_track(f, e)
print("Ordonnancement :", s_core.check())
core_z3 = [str(c) for c in s_core.unsat_core()]
print("Core Z3 (brut) :", sorted(core_z3), "- taille", len(core_z3))

# MUS irreductible
mus = compute_mus(precedences)
mus_labels = sorted(str(e) for f, e in mus)
print("MUS (irreductible) :", mus_labels, "- taille", len(mus_labels))
print("Etiquettes du core absentes du MUS :", sorted(set(core_z3) - set(mus_labels)))
print("-> Deux refutations peuvent etre irreductibles et de tailles differentes.")


Ordonnancement : unsat
Core Z3 (brut) : ['dead', 'e01', 'e12', 'e23', 'e34'] - taille 5
MUS (irreductible) : ['dead', 'e03', 'e34'] - taille 3
Etiquettes du core absentes du MUS : ['e01', 'e12', 'e23']
-> Deux refutations peuvent etre irreductibles et de tailles differentes.


**Lecture — deux preuves distinctes.** Le core brut peut suivre le chemin détaillé `e01`, `e12`, `e23`, `e34`, `dead`, alors que la suppression sur la liste entière trouve `e03`, `e34`, `dead`. Les trois premières arêtes impliquent ensemble `e03`, mais la réciproque est fausse : `e03` seule ne fixe pas les étapes intermédiaires.

Un core de cinq contraintes peut lui aussi être irréductible : le retrait de n'importe laquelle de ses contraintes le rend satisfiable. Le MUS de trois contraintes utilise une arête absente de ce core ; sa taille inférieure ne démontre donc pas que le core initial est réductible. Un MUS est minimal **par inclusion**, pas nécessairement de cardinal minimum. Il peut en exister plusieurs. Pour réparer la spécification entière, il faut vérifier à nouveau toutes les contraintes restantes : un autre conflit peut subsister.

### Exercice 2 — Arête transitive contre chemin détaillé
Dans le modèle de trois tâches, choisissez un MUS. Comparez-le au chemin détaillé ; le core brut dépend du solveur et de l'ordre.

**Étapes** : identifier l'implication du chemin vers l'arête transitive, proposer les étiquettes, puis vérifier UNSAT et la satisfiabilité après chaque retrait.

In [5]:
u = [Int(f"u{i}_ex2") for i in range(3)]
contraintes_ex2 = [(u[1] >= u[0] + 2, Bool("step01")),
    (u[2] >= u[1] + 2, Bool("step12")),
    (u[2] >= u[0] + 4, Bool("direct")),
    (u[2] <= u[0] + 3, Bool("limit"))]
reponse_ex2 = None  # TODO : liste des étiquettes d'un MUS
print("Exercice 2 à compléter" if reponse_ex2 is None else f"MUS valide : {verifier_mus(contraintes_ex2, reponse_ex2)}")

Exercice 2 à compléter


### Exercice 3 — Diagnostic et réparation
Retirez une étiquette du système de la section 2 puis vérifiez **toutes** les contraintes restantes. Retirer un membre d'un MUS ne suffit pas si un autre conflit subsiste.

**Étapes** : choisir une étiquette à retirer, reconstruire le système entier restant et contrôler SAT ; si le verdict reste UNSAT, expliquer quel conflit subsiste.

In [6]:
def reparation_valide(constraints, label):
    noms = [str(e) for _, e in constraints]
    if label is None or noms.count(label) != 1:
        return False
    s = Solver()
    s.add(*(f for f, e in constraints if str(e) != label))
    return s.check() == sat
reponse_ex3 = None  # TODO : étiquette dont le retrait répare tout le système
print("Exercice 3 à compléter" if reponse_ex3 is None else f"Réparation valide : {reparation_valide(precedences, reponse_ex3)}")

Exercice 3 à compléter


## 3. Ce que ce notebook a demontre

Trois lecons au-dela du core brut :
1. **Là où ça compte** - sur un système où aucun conflit n'est visible à l'œil (section 1), le core isole la combinaison coupable parmi de nombreuses contraintes innocentes -- c'est là que le diagnostic devient indispensable, pas décoratif.

2. **Irréductibilité** - le MUS est le sous-ensemble dont **chaque** membre est essentiel : le retirer rend le système satisfiable. L'algorithme deletion-based le garantit constructivement, là où le core brut de Z3 peut contenir des contraintes redondantes.

3. **Deux preuves du même UNSAT** - le core brut et le MUS sont deux refutations différentes de la même insatisfiabilité ; le MUS trouvé ici est plus court (3 contraintes contre 5 dans le core observé), sans invalider l’irréductibilité éventuelle de ce dernier. Comparer les deux, c'est mesurer à quel point un diagnostic peut être resserré.


## 4. Exercices
Les exercices répartis dans le carnet portent sur l'irréductibilité, les chemins transitifs et la réparation du système entier.

## Conclusion
Un core suffit à prouver l'insatisfiabilité. Un MUS est irréductible par inclusion ; il peut en exister plusieurs, de tailles différentes. Une réparation proposée doit être vérifiée sur le système entier.

`unsat_core()` diagnostique **un conflit**. Énumérer **tous les MUS** est une tâche distincte, potentiellement plus coûteuse : un diagnostic isolé ne décrit pas tous les conflits possibles.

## À retenir
| Objet | Garantie |
|---|---|
| Core | Sous-ensemble insatisfiable |
| MUS | Sous-ensemble insatisfiable irréductible |

## Vérifiez votre compréhension
Pourquoi un MUS n'est-il pas forcément de cardinal minimum ?
<details><summary>Réponse</summary>Un autre MUS incomparable peut comporter moins de contraintes.</details>

## Pour aller plus loin
Poursuivez avec [Z3-14 - Bit-vectors](Z3-14-BitVectors-Overflow-Python.ipynb).